In [ ]:
import mcfs
from mcfs.load_runs import load_run, load_runs

import numpy as np

import os
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

In [ ]:
gs, man, timing = load_run(
    base_load_path="/home/STORAGE/SKEWERS",
    simulation_name="TNG50-4",
    num=25,
    nspec=2,
    axis=-1,
    nbins=1024,
)

print("Loaded:", man["hdf5_path"])
print("z =", gs.red, "| NumLos =", gs.NumLos, "| nbins =", gs.nbins, "| dvbin =", gs.dvbin, "km/s")

if timing is not None:
    print("\n--- timing ---")
    print(timing)


In [ ]:
base_load_path = "/home/STORAGE/SKEWERS"
list_simulation_name = [
    "TNG50-4",
    "TNG50-3",
    "TNG50-2"
]
list_num   = [25]
list_nspec = [8, 16, 32, 64]
list_nbins = [2048]
axis = -1

gs_all, man_all, timing_all, missing = load_runs(
    base_load_path=base_load_path,
    list_simulation_name=list_simulation_name,
    list_num=list_num,
    list_nspec=list_nspec,
    axis=axis,
    list_nbins=list_nbins,
    strict=False,
)

print("Missing runs:", len(missing))
print("Example loaded keys:", gs_all.keys())

In [ ]:
from mcfs.xi1d import two_point_corr_1d
import jax.numpy as jnp

In [ ]:
def _log(msg, level="INFO", verbose=1):
    if verbose:
        print(f"[{level}] {msg}")


def _iter_runs(gs_all):
    """Yield (sim, num, nspec, nbins, gs) from nested dict gs_all[sim][num][nspec][nbins]."""
    for sim, dnum in gs_all.items():
        for num, dns in dnum.items():
            for nspec, db in dns.items():
                for nbins, gs in db.items():
                    yield sim, num, nspec, nbins, gs


def _flux_from_lines(gs, lines, *, verbose=1):
    """F = exp(-sum tau_line). lines = [(elem, ion, lam), ...]."""
    tau_sum = None
    for (elem, ion, lam) in lines:
        try:
            tau = np.asarray(gs.get_tau(elem, ion, lam))
        except Exception as e:
            raise RuntimeError(f"get_tau failed for ({elem},{ion},{lam}): {e}")
        tau_sum = tau if tau_sum is None else (tau_sum + tau)
    return np.exp(-tau_sum)


def _xi_for_flux(F, x, *, n_bins=256, include_self=False, center=True, standardize=False,
                 chunk_size=1 << 15, r_min=0.0, r_max=None):
    if r_max is None:
        r_max = float(x.max() - x.min())

    r_centers, xi = two_point_corr_1d(
        jnp.asarray(x),
        jnp.asarray(F),
        n_bins=n_bins,
        r_min=r_min,
        r_max=r_max,
        include_self=include_self,
        center=center,
        standardize=standardize,
        chunk_size=chunk_size,
    )

    r = np.asarray(r_centers)
    xi_np = np.asarray(xi)              # (Nlos, n_bins) expected
    m = np.nanmean(xi_np, axis=0)
    s = np.nanstd(xi_np, axis=0)
    return {"r": r, "xi": xi_np, "mean": m, "std": s}


def compute_xi_runs(
    gs_all,
    case_defs,
    *,
    selection=None,          # function(sim,num,nspec,nbins)->bool, or None
    xi_params=None,          # dict forwarded to _xi_for_flux
    verbose=1,
    on_error="skip",         # "skip" or "raise"
):
    """
    Compute xi for many runs and many cases.

    Returns:
      xi_all[sim][num][nspec][nbins] = {"meta":..., "cases": {case: {...}}}
      errors = [(sim,num,nspec,nbins,case,msg), ...]
    """
    if xi_params is None:
        xi_params = {}

    # Count planned jobs
    runs = []
    for sim, num, nspec, nbins, gs in _iter_runs(gs_all):
        if selection is None or selection(sim, num, nspec, nbins):
            runs.append((sim, num, nspec, nbins, gs))
    total = len(runs) * len(case_defs)

    xi_all = {}
    errors = []
    done = 0

    _log(f"Starting xi computation: {len(runs)} runs × {len(case_defs)} cases = {total} jobs", "INFO", verbose)

    for sim, num, nspec, nbins, gs in runs:
        xi_all.setdefault(sim, {}).setdefault(num, {}).setdefault(nspec, {})[nbins] = {
            "meta": {
                "red": float(gs.red),
                "NumLos": int(gs.NumLos),
                "nbins": int(gs.nbins),
                "dvbin_kms": float(gs.dvbin),
                "vmax_kms": float(gs.vmax),
            },
            "cases": {}
        }

        x = np.arange(gs.nbins, dtype=np.float64) * float(gs.dvbin)

        for case_name, lines in case_defs.items():
            done += 1
            _log(f"({done}/{total}) {sim} snap{num:03d} nspec={nspec} nbins={nbins} | case={case_name}", "INFO", verbose)

            try:
                F = _flux_from_lines(gs, lines, verbose=verbose)
                out = _xi_for_flux(F, x, **xi_params)
                xi_all[sim][num][nspec][nbins]["cases"][case_name] = out
            except Exception as e:
                msg = str(e)
                errors.append((sim, num, nspec, nbins, case_name, msg))
                _log(f"FAILED {sim} snap{num:03d} nspec={nspec} nbins={nbins} case={case_name}: {msg}", "ERROR", verbose)
                if on_error == "raise":
                    raise

    _log(f"Done. Successful entries: {sum(len(v2) for v1 in xi_all.values() for v2 in v1.values())} | errors: {len(errors)}", "INFO", verbose)
    return xi_all, errors


In [ ]:
case_defs = {
    "HI": [("H", 1, 1215)],
    "HI+SiIII": [("H", 1, 1215), ("Si", 3, 1206)],
    "total": [("H", 1, 1215), ("Si", 3, 1206), ("Si", 2, 1190), ("Si", 2, 1193)],
}

xi_params = dict(n_bins=256, include_self=False, center=True, standardize=False, chunk_size=1 << 15)

xi_all, errors = compute_xi_runs(gs_all, case_defs, xi_params=xi_params, verbose=1, on_error="skip")
print("errors:", len(errors))


In [ ]:
def plot_xi_entry(
    xi_entry,
    *,
    case_order,
    styles=None,          # dict: case -> {"color":..., "ls":...}
    r_max_plot=None,
    n_show=200,
    seed=0,
    vlines=None,          # list of {"x":..., "label":..., "c":"k","ls":"--","lw":1.5}
    title=None,
    verbose=1,
):
    meta = xi_entry["meta"]
    cases = xi_entry["cases"]

    # sanity
    for c in case_order:
        if c not in cases:
            _log(f"Case '{c}' not found in xi_entry['cases'] keys={list(cases.keys())}", "ERROR", verbose)
            return

    r = cases[case_order[0]]["r"]
    mask = np.ones_like(r, dtype=bool)
    if r_max_plot is not None:
        mask &= (r <= r_max_plot)

    rng = np.random.default_rng(seed)

    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    handles = []

    for case_name in case_order:
        d = cases[case_name]
        xi = d["xi"]
        m = d["mean"]
        s = d["std"]

        st = (styles or {}).get(case_name, {})
        color = st.get("color", None)
        ls = st.get("ls", "-")

        Nlos = xi.shape[0]
        n_use = min(n_show, Nlos)
        idx = rng.choice(Nlos, size=n_use, replace=False)

        _log(f"Plotting case={case_name} (show {n_use}/{Nlos} skewers)", "INFO", verbose)

        for i in idx:
            ax.plot(r[mask], xi[i, mask], alpha=0.10, lw=0.3, color=color, ls=ls)

        ax.plot(r[mask], m[mask], lw=2.6, color=color, ls=ls)
        ax.fill_between(r[mask], (m - s)[mask], (m + s)[mask], alpha=0.18, color=color)

        handles.append(Line2D([0], [0], color=color, lw=2.2, linestyle=ls, label=case_name))

    if vlines:
        for vl in vlines:
            ax.axvline(vl["x"], color=vl.get("c", "k"), ls=vl.get("ls", "--"), lw=vl.get("lw", 1.5))
            handles.append(Line2D([0], [0], color=vl.get("c", "k"), lw=vl.get("lw", 1.5),
                                  linestyle=vl.get("ls", "--"), label=vl.get("label", f"x={vl['x']}")))

    ax.legend(handles=handles, loc="upper right", fontsize=12, frameon=True)
    ax.set_xlabel(r"Separation $\Delta v$ [$\mathrm{km\,s^{-1}}$]")
    ax.set_ylabel(r"$\xi_F(\Delta v)$")
    ax.grid(True, alpha=0.25)

    if title is None:
        title = (f"2PCF | z={meta['red']:.3f} | NumLos={meta['NumLos']} | "
                 f"nbins={meta['nbins']} | dvbin={meta['dvbin_kms']:.3g} km/s")
    ax.set_title(title)

    fig.tight_layout()
    plt.show()

In [ ]:
entry = xi_all["TNG50-2"][25][64][2048]

styles = {
    "HI": {"color": "royalblue", "ls": "-"},
    "HI+SiIII": {"color": "limegreen", "ls": "--"},
    "total": {"color": "crimson", "ls": ":"},
}

c_kms = 299792.458
lam_lya = 1215.67
lam_si3 = 1206.52
dv_peak = c_kms * np.log(lam_lya / lam_si3)

vlines = [{"x": dv_peak, "label": rf"$\Delta v_{{\rm Ly\alpha-SiIII}}\approx{dv_peak:.0f}$ km/s"}]

plot_xi_entry(
    entry,
    case_order=["HI", "HI+SiIII", "total"],
    styles=styles,
    r_max_plot=8000.0,
    n_show=200,
    seed=0,
    vlines=vlines,
    verbose=1,
)


In [ ]:
# -----------------------
# Select which run to plot
# -----------------------
sim   = "TNG50-2"
num   = 25
nspec = 8
nbins = 2048

entry = xi_all[sim][num][nspec][nbins]   # must exist

# -----------------------
# Plot settings
# -----------------------
case_order = ["HI", "HI+SiIII", "total"]

colors = {"HI": "royalblue", "HI+SiIII": "limegreen", "total": "crimson"}
lss    = {"HI": "-",         "HI+SiIII": "--",        "total": ":"}

r_max_plot = 8000.0
n_show = 200
seed = 0

# Expected Lyα–SiIII misassignment separation (km/s)
c_kms = 299792.458
lam_lya = 1215.67
lam_si3 = 1206.52
dv_peak = c_kms * np.log(lam_lya / lam_si3)

# -----------------------
# Extract arrays
# -----------------------
cases = entry["cases"]
meta  = entry["meta"]

r = cases[case_order[0]]["r"]
mask = (r >= 0.0) & (r <= r_max_plot)

rng = np.random.default_rng(seed)

# -----------------------
# Plot
# -----------------------
fig, ax = plt.subplots(figsize=(8.5, 5.0))

handles = []
for name in case_order:
    xi = cases[name]["xi"]     # (Nlos, n_bins)
    m  = cases[name]["mean"]   # (n_bins,)
    s  = cases[name]["std"]    # (n_bins,)

    Nlos = xi.shape[0]
    idx = rng.choice(Nlos, size=min(n_show, Nlos), replace=False)

    col = colors[name]
    ls  = lss[name]

    for i in idx:
        ax.plot(r[mask], xi[i, mask], alpha=0.10, lw=0.3, color=col, ls=ls)

    ax.plot(r[mask], m[mask], lw=2.6, color=col, ls=ls)
    ax.fill_between(r[mask], (m - s)[mask], (m + s)[mask], alpha=0.18, color=col)

    handles.append(Line2D([0], [0], color=col, lw=2.2, ls=ls, label=name))

# vertical marker
ax.axvline(dv_peak, color="k", ls="--", lw=1.5)
handles.append(Line2D([0], [0], color="k", lw=1.5, ls="--",
                      label=rf"$\Delta v_{{\rm Ly\alpha-SiIII}}\approx {dv_peak:.0f}$ km/s"))

ax.legend(handles=handles, loc="upper right", fontsize=12, frameon=True)

ax.set_xlabel(r"Separation $\Delta v$ [$\mathrm{km\,s^{-1}}$]")
ax.set_ylabel(r"$\xi_F(\Delta v)$")
ax.grid(True, alpha=0.25)

ax.set_title(f"{sim} snap{num:03d} | nspec={nspec} nbins={nbins} | z={meta['red']:.3f}")

fig.tight_layout()
plt.show()

In [ ]:
# ---- select run from nested dicts ----
sim   = "TNG50-2"
num   = 25
nspec = 2
nbins = 2048

gs = gs_all[sim][num][nspec][nbins]

# ---- choose which absorption line to compute xi for ----
elem, ion, lam = ("H", 1, 1215)     # HI Lyα
tau = np.asarray(gs.get_tau(elem, ion, lam))   # (Nlos, nbins)

# ---- build LOS coordinate (velocity coordinate in km/s) ----
# pixel coordinate is the same for all skewers, so shape should be (nbins,)
LoS = np.arange(gs.nbins, dtype=np.float64) * float(gs.dvbin)  # [km/s]
# If your two_point_corr_1d expects x shape (nbins,) this is correct.
# If it expects (Nlos, nbins), use: LoS2 = np.tile(LoS[None, :], (tau.shape[0], 1))

# ---- 2PCF settings ----
n_bins = 256
r_min = 0.0
r_max = float(LoS.max() - LoS.min())
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

# ---- compute 2PCF for flux F = exp(-tau) ----
r_centers, xi = two_point_corr_1d(
    jnp.asarray(LoS),
    jnp.asarray(np.exp(-tau)),
    n_bins=n_bins,
    r_min=r_min,
    r_max=r_max,
    include_self=include_self,
    center=center,
    standardize=standardize,
    chunk_size=chunk_size,
)

In [ ]:

# -----------------------
# Select the loaded run
# -----------------------
sim   = "TNG50-2"
num   = 25
nspec = 2
nbins = 2048
axis  = -1

gs = gs_all[sim][num][nspec][nbins]

# -----------------------
# Load tau fields
# -----------------------
tau_HI    = np.asarray(gs.get_tau("H",  1, 1215))  # HI Lyα
tau_SiIII = np.asarray(gs.get_tau("Si", 3, 1206))  # SiIII 1206
tau_SiIIa = np.asarray(gs.get_tau("Si", 2, 1190))  # SiII 1190
tau_SiIIb = np.asarray(gs.get_tau("Si", 2, 1193))  # SiII 1193

# Flux cases
F_HI        = np.exp(-tau_HI)
F_HI_SiIII  = np.exp(-(tau_HI + tau_SiIII))
F_total     = np.exp(-(tau_HI + tau_SiIII + tau_SiIIa + tau_SiIIb))

# -----------------------
# LOS coordinate (km/s)
# -----------------------
x = np.arange(gs.nbins, dtype=np.float64) * float(gs.dvbin)  # [km/s]
r_min = 0.0
r_max = float(x.max() - x.min())

# -----------------------
# 2PCF settings
# -----------------------
n_bins = 256
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

def compute_xi(F):
    r_centers, xi = two_point_corr_1d(
        jnp.asarray(x),
        jnp.asarray(F),
        n_bins=n_bins,
        r_min=r_min,
        r_max=r_max,
        include_self=include_self,
        center=center,
        standardize=standardize,
        chunk_size=chunk_size,
    )
    r = np.asarray(r_centers)
    xi_np = np.asarray(xi)  # (Nlos, n_bins)
    m = np.nanmean(xi_np, axis=0)
    s = np.nanstd(xi_np, axis=0)
    return r, xi_np, m, s

# Compute all three
r, xi_HI,       m_HI,       s_HI       = compute_xi(F_HI)
_, xi_HI_SiIII, m_HI_SiIII, s_HI_SiIII = compute_xi(F_HI_SiIII)
_, xi_total,    m_total,    s_total    = compute_xi(F_total)

print("Shapes:", xi_HI.shape, xi_HI_SiIII.shape, xi_total.shape, "| r:", r.shape)

# -----------------------
# Expected Lyα–SiIII "contamination peak" separation
# (velocity-space shift from wavelength misassignment)
# -----------------------
c_kms = 299792.458
lam_lya  = 1215.67
lam_si3  = 1206.52
dv_peak_kms = c_kms * np.log(lam_lya / lam_si3)   # ~2271 km/s

# If you want an approximate Å-axis interpretation (small Δλ/λ):
# Δλ ≈ λ_ref * (Δv/c). Using λ_ref = λ_Lyα:
dl_peak_A_approx = lam_lya * (dv_peak_kms / c_kms)

# -----------------------
# Plot: thin skewers + mean ± std
# -----------------------
fig, ax = plt.subplots(figsize=(8.5, 5.0))

# choose a subset of skewers to plot as faint lines
rng = np.random.default_rng(0)
Nshow = min(200, xi_HI.shape[0])
idx = rng.choice(xi_HI.shape[0], size=Nshow, replace=False)

# Optionally limit x-range for readability
mask = (r >= 0.0) & (r <= 8000.0)   # adjust as you like

# Case 1: HI only
for i in idx:
    ax.plot(r[mask], xi_HI[i, mask], alpha=0.12, linewidth=0.3, color="royalblue", ls="-")
ax.plot(r[mask], m_HI[mask], linewidth=2.6, color="royalblue", ls="-")
ax.fill_between(r[mask], (m_HI - s_HI)[mask], (m_HI + s_HI)[mask], alpha=0.18, color="royalblue")

# Case 2: HI + SiIII
for i in idx:
    ax.plot(r[mask], xi_HI_SiIII[i, mask], alpha=0.10, linewidth=0.3, color="limegreen", ls="--")
ax.plot(r[mask], m_HI_SiIII[mask], linewidth=2.6, color="limegreen", ls="--")
ax.fill_between(r[mask], (m_HI_SiIII - s_HI_SiIII)[mask], (m_HI_SiIII + s_HI_SiIII)[mask],
                alpha=0.18, color="limegreen")

# Case 3: Total (HI + SiIII + SiII doublet)
for i in idx:
    ax.plot(r[mask], xi_total[i, mask], alpha=0.08, linewidth=0.3, color="crimson", ls=":")
ax.plot(r[mask], m_total[mask], linewidth=2.6, color="crimson", ls=":")
ax.fill_between(r[mask], (m_total - s_total)[mask], (m_total + s_total)[mask],
                alpha=0.16, color="crimson")

# Expected peak (velocity separation)
ax.axvline(dv_peak_kms, color="k", ls="--", lw=1.5)

# Legend
h1 = Line2D([0], [0], color="royalblue", lw=2.2, linestyle="-",  label=r"HI (Ly$\alpha$) only")
h2 = Line2D([0], [0], color="limegreen", lw=2.2, linestyle="--", label=r"HI + SiIII")
h3 = Line2D([0], [0], color="crimson",   lw=2.2, linestyle=":",  label=r"Total (HI + SiIII + SiII)")
h4 = Line2D([0], [0], color="k",         lw=1.5, linestyle="--",
            label=rf"$\Delta v_{{\rm Ly\alpha-SiIII}} \approx {dv_peak_kms:.0f}$ km/s")

ax.legend(handles=[h1, h2, h3, h4], loc="upper right", fontsize=12, frameon=True)

ax.set_xlabel(r"Separation $\Delta v$ [$\mathrm{km\,s^{-1}}$]")
ax.set_ylabel(r"$\xi_F(\Delta v)$")
ax.grid(True, alpha=0.25)

ax.set_title(f"2PCF of flux | {sim} snap{num:03d} | nspec={nspec} | nbins={nbins} | center={center}")

fig.tight_layout()
plt.show()

print(f"Expected Lyα–SiIII separation: Δv = {dv_peak_kms:.2f} km/s")
print(f"Small-Δλ approximation (using λ_Lyα): Δλ ≈ {dl_peak_A_approx:.2f} Å  (close to 1215.67-1206.52 = {lam_lya-lam_si3:.2f} Å)")


In [ ]:


# 2PCF settings
n_bins = 256
r_min = 0.0
r_max = float(x.max() - x.min())
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

# Compute 2PCF
r_centers, xi = two_point_corr_1d(
    jnp.asarray(LoS), jnp.exp(-tau),
    n_bins=n_bins, r_min=r_min, r_max=r_max, include_self=include_self,
    center=center, standardize=standardize, chunk_size=chunk_size,
)